# Fitting the Keeling curve — regression models
**Introduction to Climate Modeling · November 19, 2026**

How do we learn a function from data, and how do we judge its predictions?

We will fit five models to the Mauna Loa CO₂ record: a line through two points, a least-squares line, a quadratic, a quadratic with an annual cycle, and a neural network. Train each model on earlier observations, then test it on later years.

Open this notebook in the course [LEAP Jupyter environment](https://leap.2i2c.cloud/) and run the cells in order. The data-reading cell downloads the fixed classroom data from our course GitHub repository and saves a local copy. No separate data-folder download is needed. The first run needs internet access; later runs can use the saved files offline.

All code is supplied. **Make a guess or work it out, run, then explain.** Run the cells in order and write your responses at the **Guess before running** and **Discuss** prompts.

This is a model of **concentration as a function of time**. It does not simulate emissions or the carbon cycle. A good fit alone does not establish the physical cause of a change.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

## 1. Get and inspect the data

The Keeling curve records atmospheric CO₂ at **Mauna Loa**, rather than the global mean. Concentration is a dry-air mole fraction in **parts per million (ppm)**.

We use a fixed copy of the Scripps CO₂ Program's monthly record, through December 2025. Missing observations are removed without filling the gaps. The original source also appends recent daily data; we exclude that tail so the classroom observations are all monthly.

The next cell retrieves the prepared monthly data from our course GitHub repository if they are not already saved locally. [Scripps CO₂ Program](https://scrippsco2.ucsd.edu/) · Source details: `data/provenance.json`.

In [ ]:
from pathlib import Path
from urllib.request import urlopen
import hashlib

COURSE_DATA = 'https://raw.githubusercontent.com/dhruvbalwada/intro-climate-modeling-fall2026/f6e9334448fd452280d2d7fdd913762e7447fce9/labs/data/keeling/'

def course_file(local_name, remote_name, expected_sha256):
    """Download a fixed course file once; reuse the verified local copy."""
    path = Path(local_name)
    if not path.exists():
        print("Downloading", path.name, "from the course GitHub repository...")
        with urlopen(COURSE_DATA + remote_name, timeout=45) as response:
            content = response.read()
        if hashlib.sha256(content).hexdigest() != expected_sha256:
            raise ValueError("Downloaded file differs from the fixed classroom version.")
        path.parent.mkdir(parents=True, exist_ok=True)
        path.write_bytes(content)
    if hashlib.sha256(path.read_bytes()).hexdigest() != expected_sha256:
        raise ValueError(f"{path} differs from the classroom version. Rename it and rerun to download a fresh copy.")
    return path

data_file = course_file('data/mauna_loa_monthly_1958-2025.csv', 'mauna_loa_monthly_1958-2025.csv', 'd29ae206a7f59ae0ffedf2dbcda65b43d1a440bc2628fc19bdb394d76d61cf3e')
course_file('data/provenance.json', 'co2_provenance.json', '827868a02c54780340e8ea4a5e88ed86048c9cccf7e1010fae11d200b679b137')
print("Using:", data_file)

### Inspect and clean

`date` is time in decimal years; `co2` is concentration in ppm. We retain the original seasonal cycle.

In [ ]:
df = pd.read_csv(data_file)
df.head()

In [ ]:
df_co2_clean = df[["date", "co2"]].dropna().copy()
df_co2_clean = df_co2_clean[df_co2_clean["co2"] > 0].sort_values("date")

In [ ]:
T = df_co2_clean['date'].values
C = df_co2_clean['co2'].values

In [ ]:
plt.plot(T, C)
plt.xlabel('Year')
plt.ylabel('Mauna Loa CO₂ (ppm)')
plt.grid()

In [ ]:
print(f"{len(T)} monthly observations, {T.min():.2f}–{T.max():.2f}")

<a id="response-prediction"></a>

**Guess before running the fits:** Which parts of this record could a straight line capture, and which would it miss?

**Answer:** [Your answer here.]

## 2. Keep later years for testing

For prediction through time, keep the order of the observations. We train on observations **before 2017**, then evaluate on **2017–2025**, which the models have not seen during fitting.

The five model forms and the neural-network settings are specified in advance for this comparison. Repeatedly changing a model after inspecting these test years would turn them into development data; a final assessment would then need a separate untouched period.

In [ ]:
SPLIT_YEAR = 2017
is_train = T < SPLIT_YEAR
T_train, C_train = T[is_train], C[is_train]
T_test, C_test = T[~is_train], C[~is_train]
print(f"Training: {len(T_train)} observations, {T_train.min():.2f}–{T_train.max():.2f}")
print(f"Testing:  {len(T_test)} observations, {T_test.min():.2f}–{T_test.max():.2f}")

In [ ]:
plt.plot(T_train, C_train, label='Training data')
plt.plot(T_test, C_test, label='Test data')
plt.xlabel('Year')
plt.ylabel('Mauna Loa CO₂ (ppm)')
plt.grid()
plt.legend()

## 3. Choose a function and fit its parameters

### Model 1: a line through the first and last training observations

$$C(t)=at+b.$$

Here $a$ is the slope in ppm per year and $b$ is the intercept in ppm. This line uses two selected observations; it does not minimize the error over the whole training record.

In [ ]:
C_start = C_train[0]
T_start = T_train[0]

C_end = C_train[-1]
T_end = T_train[-1]

In [ ]:
print(T_start, C_start, T_end, C_end)

### Estimate the two parameters

Using the first and last training observations $(t_s,C_s)$ and $(t_e,C_e)$,

$$a=\frac{C_e-C_s}{t_e-t_s},\qquad b=C_s-at_s.$$

The next cells calculate these values and extend the fitted line into the test years.

In [ ]:
a = (C_start - C_end)/(T_start - T_end)
b = C_start - a * T_start

print(a, b)

In [ ]:
C_pred_train_Line = a*T_train + b
C_pred_test_Line = a*T_test + b

In [ ]:
plt.plot(T_train, C_train, label='Training data')
plt.plot(T_test, C_test, label='Test data')

plt.plot(T_train, C_pred_train_Line, linestyle='--', label='Line')
plt.plot(T_test, C_pred_test_Line, linestyle='--', label='Line')

plt.xlabel('Year')
plt.ylabel('Mauna Loa CO₂ (ppm)')
plt.grid()
plt.legend()

### Model 2: a least-squares line

The function still has two parameters,

$$C(t)=at+b,$$

but now we use **all training observations** to choose them. The loss is mean squared error:

$$\mathrm{MSE}=\frac{1}{n}\sum_{i=1}^{n}\left[C_i-\widehat C(t_i)\right]^2.$$

MSE has units of ppm². Scikit-learn's `LinearRegression` finds the coefficients that minimize the training sum of squared errors, which gives the same coefficients as minimizing MSE.

In [ ]:
from sklearn import linear_model

In [ ]:
reg = linear_model.LinearRegression()

In [ ]:
reg.fit(T_train.reshape(-1, 1), C_train.reshape(-1, 1))

In [ ]:
reg.coef_, reg.intercept_

In [ ]:
C_pred_train_LR = reg.predict(T_train.reshape(-1, 1))
C_pred_test_LR = reg.predict(T_test.reshape(-1, 1))

In [ ]:
plt.plot(T_train, C_train, label='Training data')
plt.plot(T_test, C_test, label='Test data')

plt.plot(T_train, C_pred_train_Line, linestyle='--', label='Line Train')
plt.plot(T_test, C_pred_test_Line, linestyle='--', label='Line Test')

plt.plot(T_train, C_pred_train_LR, linestyle='--', label='Linear regress Train')
plt.plot(T_test, C_pred_test_LR, linestyle='--', label='Linear regress Test')

plt.xlabel('Year')
plt.ylabel('Mauna Loa CO₂ (ppm)')
plt.grid()
plt.legend()

<a id="response-two-lines"></a>

**Discuss:** Both lines have two parameters. Why are their predictions different, and why can a low training error still leave a large test error?

**Answer:** [Your answer here.]

### Model 3: a quadratic

$$C(\tau)=a\tau^2+b\tau+c,\qquad \tau=t-1958.$$

There are three fitted parameters. Centering the year makes the numerical calculation better conditioned. It changes the coefficients, not the family of quadratic functions.

The inputs below contain $\tau$ and $\tau^2$. The regression also fits an intercept.

In [ ]:
T2_train = np.column_stack((T_train - 1958, (T_train - 1958)**2))
T2_test = np.column_stack((T_test - 1958, (T_test - 1958)**2))

In [ ]:
reg2 = linear_model.LinearRegression()

In [ ]:
reg2.fit(T2_train, C_train.reshape(-1, 1))

In [ ]:
C_pred_train_QR = reg2.predict(T2_train)
C_pred_test_QR = reg2.predict(T2_test)

In [ ]:
plt.plot(T_train, C_train, label='Training data')
plt.plot(T_test, C_test, label='Test data')

plt.plot(T_train, C_pred_train_LR, linestyle='--', label='Linear regress Train')
plt.plot(T_test, C_pred_test_LR, linestyle='--', label='Linear regress Test')

plt.plot(T_train, C_pred_train_QR, linestyle='--', label='Quadratic regress Train')
plt.plot(T_test, C_pred_test_QR, linestyle='--', label='Quadratic regress Test')

plt.xlabel('Year')
plt.ylabel('Mauna Loa CO₂ (ppm)')
plt.grid()
plt.legend()

### Model 4: a quadratic plus an annual cycle

$$C(\tau)=a\tau^2+b\tau+c\sin(2\pi\tau/P)+d,\qquad P=1\ \mathrm{year}.$$

We supply the one-year period and the sine phase. The fit chooses **four coefficients**, $a,b,c,d$. A model with an adjustable phase could include an additional cosine term, giving five coefficients.

This is an example of using prior knowledge to choose the function's form.

In [ ]:
tau_train = T_train - 1958
tau_test = T_test - 1958
T3_train = np.column_stack((tau_train, tau_train**2, np.sin(2*np.pi*tau_train)))
T3_test = np.column_stack((tau_test, tau_test**2, np.sin(2*np.pi*tau_test)))

In [ ]:
reg3 = linear_model.LinearRegression()

In [ ]:
reg3.fit(T3_train, C_train.reshape(-1, 1))

In [ ]:
C_pred_train_QRsine = reg3.predict(T3_train)
C_pred_test_QRsine = reg3.predict(T3_test)

In [ ]:
plt.figure(figsize=(12, 5))

plt.subplot(121)
plt.plot(T_train, C_train, label='Training data')
plt.plot(T_test, C_test, label='Test data')

plt.plot(T_train, C_pred_train_QR, linestyle='--', label='Quadratic regress Train')
plt.plot(T_test, C_pred_test_QR, linestyle='--', label='Quadratic regress Test')

plt.plot(T_train, C_pred_train_QRsine, linestyle='--', label='Quadratic regress+Sine Train')
plt.plot(T_test, C_pred_test_QRsine, linestyle='--', label='Quadratic regress+Sine Test')

plt.xlabel('Year')
plt.ylabel('Mauna Loa CO₂ (ppm)')
plt.grid()
plt.legend()

plt.subplot(122)
plt.plot(T_train, C_train, label='Training data')
plt.plot(T_test, C_test, label='Test data')

plt.plot(T_train, C_pred_train_QR, linestyle='--', label='Quadratic regress Train')
plt.plot(T_test, C_pred_test_QR, linestyle='--', label='Quadratic regress Test')

plt.plot(T_train, C_pred_train_QRsine, linestyle='--', label='Quadratic regress+Sine Train')
plt.plot(T_test, C_pred_test_QRsine, linestyle='--', label='Quadratic regress+Sine Test')

plt.xlabel('Year')
plt.ylabel('Mauna Loa CO₂ (ppm)')
plt.xlim((2010, 2026))
plt.ylim((380, 430))
plt.grid()
plt.legend()

plt.tight_layout()
plt.show()

### Model 5: a neural network

A multilayer perceptron combines fitted linear transformations with nonlinear activation functions. Schematically, a two-layer example is

$$\widehat C(\mathbf{x})=A_2\,\sigma(A_1\mathbf{x}+b_1)+b_2.$$

Our network below has three hidden layers with 50, 25 and 10 units. Its inputs are time and the same fixed annual sine used above. Thus both this model and Model 4 receive information about the annual cycle.

We scale the inputs and target using **training data only**. The predictions are then returned to ppm. The random seed makes the demonstration repeatable within a given software environment.

In [ ]:
from sklearn.neural_network import MLPRegressor
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.compose import TransformedTargetRegressor

<a id="response-network"></a>

**Guess before running:** Will the neural network's larger parameter count guarantee better predictions for the later years? Explain your guess.

**Answer:** [Your answer here.]

In [ ]:
# Two input features: time and a fixed annual sine.
X = np.column_stack((tau_train, np.sin(2*np.pi*tau_train)))
X_test = np.column_stack((tau_test, np.sin(2*np.pi*tau_test)))

network = MLPRegressor(hidden_layer_sizes=(50, 25, 10),
                       activation="relu", solver="adam",
                       max_iter=5000, tol=1e-6, random_state=0)
model = TransformedTargetRegressor(
    regressor=make_pipeline(StandardScaler(), network),
    transformer=StandardScaler(),
)
model.fit(X, C_train)

C_pred_train_NN = model.predict(X)
C_pred_test_NN = model.predict(X_test)
mlp = model.regressor_.named_steps["mlpregressor"]

In [ ]:
n_params = sum(W.size for W in mlp.coefs_) + sum(b.size for b in mlp.intercepts_)
print(f"Total trainable parameters: {n_params}")

In [ ]:
plt.figure(figsize=(12, 5))

plt.subplot(121)
plt.plot(T_train, C_train, label='Training data')
plt.plot(T_test, C_test, label='Test data')

plt.plot(T_train, C_pred_train_QRsine, linestyle='--', label='Quadratic regress+Sine Train')
plt.plot(T_test, C_pred_test_QRsine, linestyle='--', label='Quadratic regress+Sine Test')

plt.plot(T_train, C_pred_train_NN, linestyle='--', label='NN Train')
plt.plot(T_test, C_pred_test_NN, linestyle='--', label='NN Test')

plt.xlabel('Year')
plt.ylabel('Mauna Loa CO₂ (ppm)')
plt.grid()
plt.legend()

plt.subplot(122)
plt.plot(T_train, C_train, label='Training data')
plt.plot(T_test, C_test, label='Test data')

plt.plot(T_train, C_pred_train_QRsine, linestyle='--', label='Quadratic regress+Sine Train')
plt.plot(T_test, C_pred_test_QRsine, linestyle='--', label='Quadratic regress+Sine Test')

plt.plot(T_train, C_pred_train_NN, linestyle='--', label='NN Train')
plt.plot(T_test, C_pred_test_NN, linestyle='--', label='NN Test')

plt.xlabel('Year')
plt.ylabel('Mauna Loa CO₂ (ppm)')
plt.xlim((2010, 2026))
plt.ylim((380, 430))
plt.grid()
plt.legend()

plt.tight_layout()
plt.show()

## 4. Compare training and test error

Compare all five models using the same observations and the same loss. The logarithmic vertical axis makes both small and large errors visible.

Also compare the number of fitted parameters. Complexity and prediction skill are different properties.

In [ ]:
from sklearn.metrics import mean_squared_error

In [ ]:
MSE_test = {}
MSE_train = {}
MSE_test['Line'] = mean_squared_error(C_test, C_pred_test_Line)
MSE_test['LR'] = mean_squared_error(C_test, C_pred_test_LR)
MSE_test['QR'] = mean_squared_error(C_test, C_pred_test_QR)
MSE_test['QR_sine'] = mean_squared_error(C_test, C_pred_test_QRsine)
MSE_test['NN'] = mean_squared_error(C_test, C_pred_test_NN)

MSE_train['Line'] = mean_squared_error(C_train, C_pred_train_Line)
MSE_train['LR'] = mean_squared_error(C_train, C_pred_train_LR)
MSE_train['QR'] = mean_squared_error(C_train, C_pred_train_QR)
MSE_train['QR_sine'] = mean_squared_error(C_train, C_pred_train_QRsine)
MSE_train['NN'] = mean_squared_error(C_train, C_pred_train_NN)


In [ ]:
model_names = list(MSE_train)
counts = [2, 2, 3, 4, n_params]
scores = pd.DataFrame({
    "Model": ["Endpoint line", "Least-squares line", "Quadratic", "Quadratic + sine", "Neural network"],
    "Parameters": counts,
    "Training MSE (ppm²)": [MSE_train[name] for name in model_names],
    "Test MSE (ppm²)": [MSE_test[name] for name in model_names],
}).set_index("Model")
display(scores.round(3))

ax = scores[["Training MSE (ppm²)", "Test MSE (ppm²)"]].plot.bar(
    figsize=(9, 4.5), color=["#2374a5", "#c54b26"], rot=20)
ax.set(yscale="log", ylabel="Mean squared error (ppm²)",
       title="Fitting earlier years and predicting later years")
ax.grid(axis="y", alpha=0.25)
plt.tight_layout()
plt.show()

<a id="response-evaluation"></a>

**Discuss:** Which model has the smallest training error? Which has the smallest test error? Use the table to support your answer. Does this comparison explain the physical causes of the CO₂ increase, or justify extrapolating arbitrarily far into the future?

**Answer:** [Your answer here.]

## 5. Does it matter which years we hold out?

So far, we fitted observations before 2017 and tested on **2017–2025**. Those test times are beyond the training range. We are asking the models to **extrapolate**.

Now hold out **1987–1995**, a nine-year block near the middle of the record. Fit the remaining observations, including years on both sides of the gap. Predicting this middle block asks the models to **interpolate**.

Both comparisons use **701 training observations and 108 test observations**. We keep the five model forms, neural-network settings, random seed, and error measure unchanged. Each model is fitted again from scratch. The neural network's input and target scalers use only the training observations for that split.

This illustrates a difference between testing within and beyond the range of training inputs, often discussed as **in-distribution versus out-of-distribution prediction**. Time-series observations are not necessarily independent or stationary, so the middle block is not a formal IID sample. It also uses later years to help reconstruct earlier years: it is a gap-filling test, not a forecast made before those years occurred.


<a id="response-middle-prediction"></a>

**Guess before running:** Will the neural network's test error be smaller when the test years lie inside the training time range? Do you expect every model to improve? Explain your guess.

**Answer:** [Your answer here.]

In [ ]:
from sklearn.base import clone

# Hold out complete nine-year blocks, not randomly scattered months.
block_test_masks = {
    "End: 2017–2025": T >= 2017,
    "Middle: 1987–1995": (T >= 1987) & (T < 1996),
}
assert all(mask.sum() == 108 for mask in block_test_masks.values())
assert all((~mask).sum() == 701 for mask in block_test_masks.values())

u_all = T - 1958
sine_all = np.sin(2*np.pi*u_all)
# Reuse the same model settings and features as above. clone() creates a fresh,
# unfitted estimator, including fresh training-only scalers for the network.
block_templates = {
    "Least-squares line": (reg, T.reshape(-1, 1)),
    "Quadratic": (reg2, np.column_stack([u_all, u_all**2])),
    "Quadratic + sine": (reg3, np.column_stack([u_all, u_all**2, sine_all])),
    "Neural network": (model, np.column_stack([u_all, sine_all])),
}
block_predictions = {}
block_rows = []
for split_name, test_mask in block_test_masks.items():
    train_mask = ~test_mask
    first, last = np.flatnonzero(train_mask)[[0, -1]]
    slope = (C[last]-C[first])/(T[last]-T[first])
    predictions = {"Endpoint line": C[first]+slope*(T-T[first])}
    for name, (template, features) in block_templates.items():
        fitted = clone(template)
        fitted.fit(features[train_mask], C[train_mask])
        predictions[name] = np.asarray(fitted.predict(features)).ravel()
    block_predictions[split_name] = predictions
    for name, prediction in predictions.items():
        block_rows.append({"Split": split_name, "Model": name,
                           "Training MSE (ppm²)": mean_squared_error(C[train_mask], prediction[train_mask]),
                           "Test MSE (ppm²)": mean_squared_error(C[test_mask], prediction[test_mask])})

block_scores = pd.DataFrame(block_rows)
block_test_scores = block_scores.pivot(index="Model", columns="Split", values="Test MSE (ppm²)")
block_test_scores = block_test_scores.reindex(scores.index)[list(block_test_masks)]
display(block_test_scores.round(3))

ax = block_test_scores.plot.bar(figsize=(10, 4), rot=20, color=["#2673A1", "#B54C2F"])
ax.set(yscale="log", ylabel="Test mean squared error (ppm²)",
       title="Same model settings, different test periods")
ax.grid(axis="y", alpha=0.25)
plt.tight_layout()
plt.show()

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(11, 7), sharex=True, sharey=True, layout="constrained")
for ax, (split_name, test_mask) in zip(axes, block_test_masks.items()):
    ax.plot(T[~test_mask], C[~test_mask], ".", ms=2, color="0.65", label="Training observations")
    ax.plot(T[test_mask], C[test_mask], ".", ms=3, color="#D99523", label="Test observations")
    ax.axvspan(T[test_mask].min(), T[test_mask].max(), color="#D99523", alpha=0.12, zorder=-1)
    for name, color in [("Quadratic + sine", "#2673A1"), ("Neural network", "#B54C2F")]:
        ax.plot(T, block_predictions[split_name][name], color=color, lw=1.1, label=name)
    ax.set(title=split_name, ylabel="Mauna Loa CO₂ (ppm)")
    ax.legend(fontsize=9, loc="upper left")
axes[-1].set_xlabel("Year")
plt.show()

<a id="response-middle-interpretation"></a>

**Discuss:** Does the neural network's ranking change between the two tests? Use the table and plots to explain any differences. Which split better answers our original question about predicting later observations, and why doesn't a small middle-block error establish future prediction skill?

**Answer:** [Your answer here.]

The test periods cover different years as well as different positions relative to the training inputs. Treat the comparison as one illustration, not a universal ranking of algorithms. Repeatedly changing the models after inspecting these blocks would make them development data; a final evaluation would need another untouched period.

## Sources and reuse

Adapted from Dhruv Balwada's [original Keeling-curve Colab exercise](https://colab.research.google.com/drive/1rcy60oLVwHnzgB952nSGudL5fR_4foOB).

Data: **Scripps CO₂ Program**, Scripps Institution of Oceanography, UC San Diego. The fixed source version, processing choices and checksums are recorded in `data/provenance.json`. The source record is subject to calibration and quality-control revisions.

Suggested source citation: C. D. Keeling and colleagues (2005), *Atmospheric CO₂ and ¹³CO₂ exchange with the terrestrial biosphere and oceans from 1978 to 2000: observations and carbon cycle implications*, in *A History of Atmospheric CO₂ and Its Effects on Plants, Animals, and Ecosystems*, pp. 83–113.

[Scripps CO₂ data](https://scrippsco2.ucsd.edu/data/atmospheric_co2/) · [Scikit-learn: linear regression](https://scikit-learn.org/stable/modules/generated/sklearn.linear_model.LinearRegression.html) · [Scikit-learn: neural-network regression](https://scikit-learn.org/stable/modules/generated/sklearn.neural_network.MLPRegressor.html)


Model-evaluation background: [Scikit-learn cross-validation guidance](https://scikit-learn.org/stable/modules/cross_validation.html), including training-only transformations and time-series splits.

## Next: emissions-driven models

This is **Notebook 1 of 2**. We learned concentration as a function of time. In [Notebook 2: emissions-driven models](https://dhruvbalwada.github.io/intro-climate-modeling-fall2026/labs/keeling_curve_dynamics.html), we keep quadratic-plus-sine as a reference, introduce a one-box model through the EBM analogy, and explore a richer carbon response.

The second notebook runs independently and supplies its own data. It evaluates through **2024**, the last year in the bundled historical emissions record, and refits the baseline on the same pre-2017 observations. This notebook retains the station observations through **2025**, so its final test-error number uses one additional year.
